$QK^T$, resulting in an $S \times S$ matrix. If your sequence has 10,000 tokens, that matrix has 100 million elements. This makes long-sequence processing very expensive.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.3 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [23]:
# ✏️ YOUR IMPLEMENTATION HERE

def linear_attention(Q, K, V):
    num = torch.matmul(F.elu(Q)+1, torch.matmul(torch.transpose(F.elu(K)+1, 1, 2), V))
    den = torch.matmul(F.elu(Q)+1, torch.transpose(torch.sum(F.elu(K)+1, dim = 1, keepdim=True), 1, 2)) # sum along seq_len to form global key
    return num/den
    pass  # Replace ths

In [24]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [25]:
torch.matmul(F.elu(Q)+1, torch.matmul(torch.transpose(F.elu(K)+1, 1, 2), V))

tensor([[[-5.1938e+01, -1.9063e+01, -4.7469e+01, -8.5112e+01, -5.2302e+01,
           5.0370e+01,  3.8680e+01, -4.0552e+01,  1.8285e+01, -3.8708e+01,
          -5.0901e+01, -8.3139e+01,  1.2915e+02, -8.3042e+00,  2.8288e+01,
          -5.4814e+01, -3.2576e+00, -5.4603e+01, -1.6692e+01,  1.1493e+00,
          -6.2248e+01, -9.8063e+01,  9.7043e-01,  2.0547e+01, -7.5678e+01,
          -1.6030e+01, -9.4637e+01,  6.4546e+01,  5.0223e+01,  1.6481e+00,
           7.9500e+01, -5.4168e+01],
         [-6.4506e+01, -1.9511e+01, -6.7202e+01, -1.0773e+02, -5.9029e+01,
           6.7794e+01,  4.7137e+01, -4.5751e+01,  2.7471e+01, -6.2430e+01,
          -7.2191e+01, -9.4395e+01,  1.8439e+02,  1.6065e-01,  3.2511e+01,
          -6.6269e+01,  1.8824e+01, -5.8008e+01,  1.7545e+00,  2.6933e+01,
          -7.9911e+01, -1.2440e+02, -1.0348e+01,  1.4524e+01, -1.1206e+02,
          -1.3342e+01, -1.3334e+02,  6.0389e+01,  5.2905e+01, -1.2352e+01,
           9.0225e+01, -6.9860e+01],
         [-5.8694e+01, -1.

In [26]:
torch.sum(F.elu(K)+1, dim = 1, keepdim=True).shape

torch.Size([1, 1, 16])

In [27]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.0ms)
  ✅ [2/4] No NaN or Inf (6.5ms)
  ✅ [3/4] Gradient flow (24.0ms)
  ✅ [4/4] Runs fast on long sequences (linear complexity) (44.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (76.1ms total)
  Progress saved. Run status() to see your dashboard.

